# data conversion scripts

converts everything to zarr files

performs hotpixel correction too

# cSPAD quanta data formats to zarr/hdf5

Known formats (individual `.bin`/`.mat` files, or directories of them) to zarr.

You should generally do this for compression.

In [ ]:
"""
cSPAD raw data folder (a bunch of .bin files instead of a single one) to Zarr format.
"""
%load_ext autoreload
%autoreload 2
%matplotlib inline
from pathlib import Path

import numpy as np

import spadlib.io as spio
from spadlib.processing import correct_hotpixels
from sptflux.utils import get_path_stem

datanames = ["/scratch/jeyan/data/uwb-data/quanta-megapix/paintball-moonlight-1.h5"]
perform_hotpixel_correction = True
bayer = True
H = 1024
W = 1024
fps = 57_000
for dataname in datanames:
    dataname = Path(dataname)
    print(f"Processing {dataname}...")
    frames = spio.read_quanta_auto(dataname, H=H, W=W, load_data=True)

    if perform_hotpixel_correction:
        out_path = dataname.parent / f"{get_path_stem(dataname)}-hotpixel_corrected.h5"
    else:
        out_path = dataname.parent / f"{get_path_stem(dataname)}.h5"

    if perform_hotpixel_correction:
        print("Correcting hot pixels...")
        frames = correct_hotpixels(frames, bayer=bayer, hp_threshold=2.0, kernel_size=3)
    spio.write_quanta_hdf5(out_path, frames, fps=fps)
    del frames

# npy file from the wision data

In [ ]:
"""
npy file from the wision data or something to Zarr format.
"""
%load_ext autoreload
%autoreload 2
%matplotlib inline
from pathlib import Path
import numpy as np

from spadlib.io import read_quanta_dir, write_quanta_zarr
from spadlib.processing import correct_hotpixels

dataname = "lighter"
data_path = Path(f"data/quanta/{dataname}.npy")
a = np.load(data_path)
A = np.unpackbits(a, axis=2)[:, ::-1, ::-1]

out_path = data_path.parent / f"{dataname}.zarr"

A_sliced = A[30000:96000, :, 16:]
A_sliced = correct_hotpixels(A_sliced)
write_quanta_zarr(out_path, A_sliced, fps=1 / 1.05e-5)

# Asynchronous SPAD data conversion

In [ ]:
"""
Asynchronous SPAD data conversion to Zarr format.
"""
%load_ext autoreload
%autoreload 2
%matplotlib inline
import numpy as np
from pathlib import Path

from spadlib.io import read_async_spad_dir, write_async_spad_zarr

T_exp = 0.1
W = 138
H = 91

datapath = Path("data/async-spad/AC_fan/G")
if not datapath.is_dir():
    raise ValueError()
outpath = datapath.parent / f"{datapath.name}.zarr"

points, pixel_timeseries = read_async_spad_dir(datapath, h=H, w=W)

write_async_spad_zarr(outpath, pixel_timeseries, T_exp=T_exp)